# 📘 Document Question Answering System (RAG) using  Gemini API

---

## Step 1: Setup - imports and Gemini config

In [59]:
!pip install google-generativeai PyPDF2 python-docx numpy -q

In [72]:
import os
import numpy as np
import google.generativeai as genai
from PyPDF2 import PdfReader
from docx import Document

api_key = "my api key"
genai.configure(api_key=api_key)

embed_model = "models/gemini-embedding-001"
chat_model = genai.GenerativeModel("gemini-flash-latest")

print("setup done")

setup done


## Step 2: Document ingestion function

In [73]:
def load_document(path):
    if path.endswith(".pdf"):
        reader = PdfReader(path)
        text = "".join(page.extract_text() or "" for page in reader.pages)
    elif path.endswith(".docx"):
        doc = Document(path)
        text = "\n".join(p.text for p in doc.paragraphs)
    elif path.endswith(".txt"):
        text = open(path, "r", encoding="utf-8").read()
    else:
        print("file type not supported")
        text = ""
    return text

## Step 3: Chunking function

In [74]:
def make_chunks(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

## Step 4: Embedding function (Gemini)

In [75]:
def get_embedding(text):
    return genai.embed_content(model=embed_model, content=text)["embedding"]

## Step 5: Vector similarity and retrieval

In [76]:
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_top_chunks(question, chunks, chunk_embeddings, top_k=3):
    q_emb = get_embedding(question)
    scores = [cosine_sim(q_emb, emb) for emb in chunk_embeddings]
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [chunks[i] for i in ranked], [scores[i] for i in ranked]

## Step 6: Answer generation (Gemini)

In [77]:
def generate_answer(question, top_chunks, top_scores, threshold=0.5):
    if max(top_scores) < threshold:
        return "This information is not present in the uploaded document."
    context = "\n\n".join(top_chunks)
    prompt = (f"Answer the question using only the context below. "
              f"If the answer is not in the context, say it is not found in the document.\n\n"
              f"Context:\n{context}\n\nQuestion: {question}")
    return chat_model.generate_content(prompt).text

## Step 7: Choose or Upload a Document

In [78]:
default_path = "sample_docs/sample.txt"
user_path = input("Enter path of your pdf/docx/txt file (or press Enter to use sample doc): ").strip()
file_path = user_path if user_path else default_path

if not os.path.exists(file_path):
    print("file not found, falling back to sample document")
    file_path = default_path

print("using file:", file_path)

Enter path of your pdf/docx/txt file (or press Enter to use sample doc):  C:\Users\Administrator\Downloads\solar_system.pdf


using file: C:\Users\Administrator\Downloads\solar_system.pdf


## Step 8: Run ingestion, chunking, and embedding

In [79]:
raw_text = load_document(file_path)
chunks = make_chunks(raw_text)
chunk_embeddings = [get_embedding(c) for c in chunks]

print("total characters loaded:", len(raw_text))
print("total chunks made:", len(chunks))
print("embedding size:", len(chunk_embeddings[0]))

total characters loaded: 2327
total chunks made: 6
embedding size: 3072


## Step 9: Testing it out with a few questions

In [80]:
import time
test_questions = [
    "What is the Sun mostly made of?",
    "Which planet is known as the Red Planet and why?",
    "How many known moons does Jupiter have?",
    "Why does Uranus have a blue-green color?",
    "What year was Pluto reclassified as a dwarf planet?",
    "What is the closest planet to the Sun?",
    "What is the capital of France?"
]

for q in test_questions:
    top_chunks, top_scores = retrieve_top_chunks(q, chunks, chunk_embeddings)
    answer = generate_answer(q, top_chunks, top_scores)
    print("Q:", q)
    print("top similarity score:", round(max(top_scores), 3))
    print("A:", answer)
    print("---")
    time.sleep(13)

Q: What is the Sun mostly made of?
top similarity score: 0.733
A: Based on the provided context, the Sun is made mostly of hydrogen and helium gas.
---
Q: Which planet is known as the Red Planet and why?
top similarity score: 0.707
A: Based on the provided context, Mars is known as the Red Planet because of the iron oxide on its surface.
---
Q: How many known moons does Jupiter have?
top similarity score: 0.695
A: Based on the provided context, Jupiter has at least 79 known moons.
---
Q: Why does Uranus have a blue-green color?
top similarity score: 0.681
A: Based on the provided context, Uranus has a blue-green color because of methane in its atmosphere.
---
Q: What year was Pluto reclassified as a dwarf planet?
top similarity score: 0.684
A: Based on the provided context, Pluto was reclassified as a dwarf planet in 2006.
---
Q: What is the closest planet to the Sun?
top similarity score: 0.737
A: Based on the provided context, Mercury is the closest planet to the Sun.
---
Q: What is 

## Step 10: System metrics report

In [81]:
print("chunk size:", 500, "| overlap:", 50, "| total chunks:", len(chunks))
print("embedding model:", embed_model, "| dimension:", len(chunk_embeddings[0]))
print("llm:", "gemini-flash-latest", "| vector store: in-memory list + cosine similarity")
print("retrieval top_k:", 3)

chunk size: 500 | overlap: 50 | total chunks: 6
embedding model: models/gemini-embedding-001 | dimension: 3072
llm: gemini-flash-latest | vector store: in-memory list + cosine similarity
retrieval top_k: 3


## Step 11: Ask your own question

In [82]:
question = "How can I get a job in Celebal Technologies?"

top_chunks, top_scores = retrieve_top_chunks(question, chunks, chunk_embeddings)
answer = generate_answer(question, top_chunks, top_scores)

print("Question:", question)
print("Answer:", answer)

Question: How can I get a job in Celebal Technologies?
Answer: This information is not present in the uploaded document.
